In [0]:
# ALERTS TABLE
CATALOG = "airbnb_obs"

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {CATALOG}.monitoring.alerts (
    alert_ts       TIMESTAMP,
    severity       STRING,      -- INFO / WARNING / CRITICAL
    source_table   STRING,      -- which table triggered it
    rule_name      STRING,      -- which rule failed
    message        STRING,      -- human-readable alert text
    failed_records BIGINT,
    resolved       BOOLEAN      -- tracking flag; starts false
) USING DELTA
""")
print("alerts table ready")

In [0]:
#ALERT GENERATOR

from pyspark.sql import functions as F
from datetime import datetime, timezone

def generate_alerts():
    # pull the latest DQ verdicts that FAILED
    fails = (spark.table(f"{CATALOG}.monitoring.v_current_dq_state")
                  .filter(F.col("status") == "FAIL")
                  .collect())

    if not fails:
        print("✔ no failures — nothing to alert on")
        return

    rows_to_write = []
    for row in fails:
        total = row["total_records"] or 0
        failed = row["failed_records"] or 0
        pct = (failed / total) if total else 0

        # severity logic: bigger share of bad records = louder alert
        if   pct >= 0.10: severity = "CRITICAL"
        elif pct >= 0.01: severity = "WARNING"
        else:             severity = "INFO"

        msg = (f"[{severity}] {row['table_name']}.{row['rule_name']} failed: "
               f"{failed} of {total} records ({pct*100:.2f}%)")

        rows_to_write.append((
            datetime.now(timezone.utc),
            severity,
            row["table_name"],
            row["rule_name"],
            msg,
            int(failed),
            False,               # resolved starts false
        ))

    cols = ["alert_ts","severity","source_table","rule_name",
            "message","failed_records","resolved"]
    (spark.createDataFrame(rows_to_write, cols)
          .write.mode("append")
          .saveAsTable(f"{CATALOG}.monitoring.alerts"))

    print(f"⚠ wrote {len(rows_to_write)} alert(s)")
    for r in rows_to_write:
        print("   ", r[4])   # print each message

generate_alerts()

In [0]:
%sql
-- CHECK

SELECT alert_ts, severity, source_table, rule_name, failed_records, resolved
FROM airbnb_obs.monitoring.alerts
ORDER BY alert_ts DESC;